In [1]:
#initialisation 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
#import contextily as ctx      #marchait pas
import seaborn as sns          #ajout par rapport au main

df = pd.read_csv("../output/df_modele_musees.csv")
df.head()

,id_patrimostat,id_museofile,dateappellation,ferme,anneefermeture,ville,codeInseeCommune,annee,payant,gratuit,...,annee_creation,latitude,longitude,total_frequentation,age_musee,age_musee_missing,total_t_1,croissance_total,has_excel,est_idf
0,6702101,M0001,01/02/2003,NON,NaN,BARR,67021,2014,1865.0,2685.0,...,1960.0,48.410166,7.451102,4550.0,54.0,0,NaN,NaN,1,0
1,6702101,M0001,01/02/2003,NON,NaN,BARR,67021,2015,1874.0,1934.0,...,1960.0,48.410166,7.451102,3808.0,55.0,0,4550.0,-0.163077,1,0
2,6702101,M0001,01/02/2003,NON,NaN,BARR,67021,2016,1705.0,1409.0,...,1960.0,48.410166,7.451102,3114.0,56.0,0,3808.0,-0.182248,1,0
3,6702101,M0001,01/02/2003,NON,NaN,BARR,67021,2017,1163.0,1281.0,...,1960.0,48.410166,7.451102,2444.0,57.0,0,3114.0,-0.215157,1,0
4,6702101,M0001,01/02/2003,NON,NaN,BARR,67021,2018,1249.0,2341.0,...,1960.0,48.410166,7.451102,3590.0,58.0,0,2444.0,0.468903,1,0


In [2]:
#NETTOYAGE DES CATÉGORIES

def clean_text(x):
    if pd.isna(x):
        return None
    x = x.lower().strip()
    x = x.rstrip(".")   
    x = x.replace(";", "/").replace(",", "/").replace("|", "/").replace("."," /")
    x = " ".join(x.split())  # retire espaces multiples
    return x

df["categorie_clean"] = df["categorie"].apply(clean_text)

map_cat = {
    "ecomusée": "écomusée",
    "maison d'artiste": "maison musée",
    "maison d'illustre": "maison musée",
    "maison des illustres": "maison musée",
    "musée en zone rurale": "musée en milieu rural",
    "musée de site : site archéologique": "musée de site",
    "musée de site : carreau de mine": "musée de site",
    "architecture contemporaine remarquable (extension)": "architecture contemporaine remarquable",
    "musée de site : usine": "musée de site",
    "musée de site / musée en milieu rural" : "musée de site",
    "écomusée / musée en milieu rural": "écomusée",
    "écomusée / musée en zone rurale": "écomusée",
    "musée de site / musée en zone rurale" : "musée de site",
    "maison musée / maison des illustres" : "maison musée",
    "musée littéraire / musée en milieu rural": "musée littéraire",
    "musée de site / jardin remarquable": "jardin remarquable",
    "domaine national / jardin remarquable": "jardin remarquable",
    "maison des illustres / musée en milieu rural": "maison musée",
    "maison musée / maison des illustres / musée en milieu rural": "maison musée",
    "musée de plein air / musée en milieu rural": "musée de plein air",
    "musée de site / architecture contemporaine remarquable": "architecture contemporaine remarquable",
    "musée d'art sacré / musée en milieu rural": "musée d'art sacré",
    "musée de site / maison d'artiste" : "maison musée",
    "musée de site / maison des illustres" : "maison musée",
    "musée de site / site archéologique / musée en milieu rural": "musée de site",
    "ecomusée / musée de plein air / musée de site" : "écomusée",
    "musée de site / musée en zone rurale / site archéologique" : "musée de site",
    "ecomusée / musée de plein air" : "écomusée", 
    "ecomusée / musée de site" : "écomusée",
    "musée de site / maison musée / maison des illustres" : "maison musée",
    "musée de site / musée littéraire" : "musée littéraire",
    "musée de site / domaine national": "domaine national",
    "écomusée / musée de site / musée en zone rurale" : "écomusée",
    "architecture contemporaine remarquable / musée en milieu rural" :"architecture contemporaine remarquable",
    "musée de plein air / musée de site":"musée de site",
    "musée de plein air / maison musée / maison des illustres / musée en milieu rural" : "maison musée",
    "écomusée / musée de plein air / musée de site / musée en milieu rural" :"écomusée",
    "musée de plein air / musée de site / musée en zone rurale":"musée de site",
    "musée de site / musée littéraire / maison des illustres (2017)": "maison musée littéraire",
    "musée littéraire / architecture contemporaine remarquable": "musée littéraire",
    "musée de site / musée littéraire / maison des illustres / musée en milieu rural" : "maison musée littéraire",
    "musée de site / maison musée / musée littéraire / maison des illustres / musée en zone rurale": "maison musée littéraire",
    "maison musée / maison d'artiste / musée littéraire" : "maison musée littéraire",
    "écomusée / musée de site / musée en milieu rural" : "écomusée",
    "écomusée / musée de plein air / musée de site / jardin remarquable / musée en zone rurale" : "écomusée",
    "maison musée / maison des illustres / musée en zone rurale" : "maison musée",
    "musée littéraire / maison des illustres / musée en zone rurale": "maison musée littéraire",
    "maison d'artiste / musée littéraire / maison des illustres": "maison musée",
    "maison musée / maison des illustres / jardin remarquable" : "maison musée",
    "maison musée / / maison des illustres / musée en milieu rural" : "maison musée",
    "musée de mode/ maison des illustres" : "musée de mode",
    "musée de site / maison d'artiste / musée d'art sacré" : "maison musée",
    "jardin remarquable / musée en milieu rural" : "jardin remarquable",
    "maison musée / musée littéraire / maison des illustres / jardin remarquable" : "maison musée littéraire",
    "musée de site / maison musée / musée littéraire / maison des illustres" : "maison musée littéraire",
    "maison musée / musée littéraire" : "maison musée littéraire",
    "musée littéraire / maison des illustres / musée en milieu rural" : "maison musée littéraire",
    "maison musée / musée littéraire / maison des illustres / musée en milieu rural" : "maison musée littéraire",
    "maison musée / musée littéraire / maison des illustres" : "maison musée littéraire",
    "musée littéraire / maison des illustres" : "maison musée littéraire"
}

df["categorie_norm"] = df["categorie_clean"].replace(map_cat)

print(df["categorie_norm"].value_counts(dropna=False).to_string())


categorie_norm
None                                      6590
musée en milieu rural                     1639
musée de site                             1098
écomusée                                   516
maison musée                               315
maison musée littéraire                    278
musée littéraire                           226
jardin remarquable                         151
architecture contemporaine remarquable     146
musée d'art sacré                          101
musée de plein air                          77
domaine national                            60
muséum                                      10
musée de mode                               10
musée de france                             10
beaux-arts                                   9
musée de société                             8
musée d'histoire                             6


In [3]:
print(df["domaine_thematique"].value_counts(dropna=False).to_string())

domaine_thematique
Archéologie                                                                                                                                                                                       680
NaN                                                                                                                                                                                               414
Histoire                                                                                                                                                                                          354
Beaux-arts                                                                                                                                                                                        337
Sciences de la nature                                                                                                                                                                        

In [4]:
#NETTOYAGE DES DOMAINES

def clean_text(x):
    if pd.isna(x):
        return None
    x = x.lower().strip()
    x = x.rstrip(".")   
    x = x.replace(";", "/").replace(",", "/").replace("|", "/").replace("."," /")
    x = " ".join(x.split())  # retire espaces multiples
    return x

df["domaine_clean"] = df["domaine_thematique"].apply(clean_text)

df["domaine_list"] = df["domaine_clean"].apply(lambda x: x.split("/") if pd.notna(x) else [])

print(df["domaine_list"].value_counts(dropna=False).to_string())

domaine_list
[archéologie]                                                                                                                                                                                                    680
[beaux-arts]                                                                                                                                                                                                     481
[]                                                                                                                                                                                                               414
[histoire]                                                                                                                                                                                                       354
[sciences de la nature]                                                                                                                